# 04 - LLM con Few-Shot Prompting

Probamos el enfoque LLM (Claude / OpenAI) con few-shot prompting para ABSA.

**Requisitos:**
- Variable de entorno `ANTHROPIC_API_KEY` o `OPENAI_API_KEY`.
- Paquetes `anthropic` u `openai` instalados.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.aspects.extractor import extract_aspects, load_spacy_model
from src.data.loader import load_reviews
from src.llm.prompting import analyze_with_llm, build_default_client

## 1. Construir cliente

In [ ]:
client = build_default_client()
type(client).__name__

## 2. Extraer aspectos y analizar

In [ ]:
df = load_reviews("../data/sample/reviews_sample.csv")
nlp = load_spacy_model()

results = []
for _, row in df.head(5).iterrows():
    text = row['text']
    aspects = extract_aspects(text, nlp)
    sentiments = analyze_with_llm(text, aspects, client)
    results.append({'text': text, 'aspects': sentiments})
results

## 3. Comparar con etiquetas del lexicón

Útil para detectar dónde el LLM y el baseline discrepan.

In [ ]:
from src.classical.lexicon import load_lexicon, score_aspect_lexicon

lex = load_lexicon()
to_lbl = lambda s: 'pos' if s > 0.1 else 'neg' if s < -0.1 else 'neu'

for r in results:
    print(r['text'])
    for asp, lbl in r['aspects'].items():
        lex_lbl = to_lbl(score_aspect_lexicon(r['text'], asp, lex))
        marker = ' ' if lex_lbl == lbl else '*'
        print(f"  {marker} {asp:>15} | LLM={lbl} | Lex={lex_lbl}")
    print()

## 4. Coste y latencia

Documentar tokens consumidos y tiempo por reseña antes de escalar a miles.